# Exercise 5: Tiled matrix multiplication: solution

Solution to Exercise 5 in `matrix_multiplication.ipynb`.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

In [ ]:
using KernelAbstractions

# Pick your hardware: :NVIDIA, :AMD, :Metal, or :CPU
DEV = :NVIDIA

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
elseif DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
else
    error("Unknown DEV = $DEV. Use :NVIDIA, :AMD, :Metal, or :CPU.")
end

In [ ]:
const TILE_DIM = 16

@kernel function tiled_matmul_kernel!(C, @Const(A), @Const(B))
    gi, gj = @index(Group, NTuple)
    i, j = @index(Local, NTuple)

    tileA = @localmem eltype(C) (TILE_DIM, TILE_DIM)
    tileB = @localmem eltype(C) (TILE_DIM, TILE_DIM)

    row = (gi - 1) * TILE_DIM + i
    col = (gj - 1) * TILE_DIM + j
    acc = zero(eltype(C))

    for t in 0:(size(A, 2) ÷ TILE_DIM - 1)
        @inbounds tileA[i, j] = A[row, t * TILE_DIM + j]
        @inbounds tileB[i, j] = B[t * TILE_DIM + i, col]

        @synchronize

        for k in 1:TILE_DIM
            @inbounds acc += tileA[i, k] * tileB[k, j]
        end

        @synchronize
    end

    @inbounds C[row, col] = acc
end

In [ ]:
matrix_size = 1024
A = rand(Float32, matrix_size, matrix_size)
B = rand(Float32, matrix_size, matrix_size)

DA = allocate(Backend, Float32, matrix_size, matrix_size)
copyto!(DA, A)
DB = allocate(Backend, Float32, matrix_size, matrix_size)
copyto!(DB, B)
DC = KernelAbstractions.zeros(Backend, Float32, matrix_size, matrix_size)

tiled! = tiled_matmul_kernel!(Backend, (TILE_DIM, TILE_DIM))
tiled!(DC, DA, DB, ndrange=size(DC))
KernelAbstractions.synchronize(Backend)

isapprox(Array(DC), A * B)